In [ ]:
# ============================================================
# SegResNet 3D - TASK03 LIVER DATASET
# Third Model
# ============================================================

import torch
import numpy as np

from tqdm import tqdm

from monai.networks.nets import SegResNet



# ============================
# DEVICE
# ============================

device="cuda" if torch.cuda.is_available() else "cpu"

print("Device:",device)



# ============================
# MODEL
# ============================


model = SegResNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,

    init_filters=16,

    dropout_prob=0.2
).to(device)



# ============================
# OPTIMIZER
# ============================

optimizer=torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)



# ============================
# TRAINING
# ============================

EPOCHS=5



for epoch in range(EPOCHS):


    model.train()


    epoch_loss=0



    loop=tqdm(train_loader)



    for img,mask in loop:


        img=img.to(device)

        mask=mask.to(device)



        pred=model(img)


        pred=torch.sigmoid(pred)



        loss=dice_loss(
            pred,
            mask
        )



        optimizer.zero_grad()


        loss.backward()


        optimizer.step()



        epoch_loss+=loss.item()



        loop.set_description(
            f"Epoch {epoch+1}/{EPOCHS}"
        )



    print(
        "Loss:",
        epoch_loss/len(train_loader)
    )



# ============================
# VALIDATION
# ============================


model.eval()


dice_scores=[]



with torch.no_grad():


    for img,mask in val_loader:


        img=img.to(device)

        mask=mask.to(device)



        pred=model(img)


        pred=torch.sigmoid(pred)


        pred=(pred>0.5).float()



        intersection=(pred*mask).sum()



        dice=(2*intersection)/(
            pred.sum()+mask.sum()+1e-5
        )



        dice_scores.append(
            dice.item()
        )



print(
    "SegResNet Validation Dice:",
    np.mean(dice_scores)
)



# ============================
# SAVE MODEL
# ============================


torch.save(
    model.state_dict(),
    "SegResNet3D_Liver.pth"
)


print(
    "Saved: SegResNet3D_Liver.pth"
)

Device: cuda


Epoch 1/5: 100%|██████████| 104/104 [02:28<00:00,  1.42s/it]


Loss: 0.878131838945242


Epoch 2/5: 100%|██████████| 104/104 [02:28<00:00,  1.43s/it]


Loss: 0.8595918416976929


...

Loss: 0.8479281205397385


Epoch 4/5: 100%|██████████| 104/104 [02:28<00:00,  1.43s/it]


Loss: 0.8348058422024434


Epoch 5/5: 100%|██████████| 104/104 [02:26<00:00,  1.41s/it]


Loss: 0.8200333187213311
SegResNet Validation Dice: 0.9085001349449158
Saved: SegResNet3D_Liver.pth


In [ ]:
!pip install -q monai

In [ ]:
DATA_DIR = "/kaggle/input/liver-cancer-multiclass-dataset/Liver_Dataset/Liver_Dataset"